In [1]:

import pandas as pd
import numpy as np

# Load the same dataset used in the forecasting notebooks
df = pd.read_csv(
    "simulated_energy_data.csv",
    parse_dates=["timestamp"]
)

# Build campus-level hourly data
hourly = (
    df.groupby("timestamp")
    .agg(
        campus_kw=("total_power_kw", "sum"),
        scheduled_rooms=("scheduled", "sum"),
        outside_temp_c=("outside_temp_c", "first"),
        day_type=("day_type", "first")
    )
    .reset_index()
)

hourly["date"] = hourly["timestamp"].dt.date
hourly["hour"] = hourly["timestamp"].dt.hour

# Extract daily peak demand and peak hour
peak_idx = hourly.groupby("date")["campus_kw"].idxmax()

daily_peak = (
    hourly.loc[peak_idx, ["date", "timestamp", "campus_kw", "hour"]]
    .rename(columns={
        "timestamp": "peak_timestamp",
        "campus_kw": "peak_kw",
        "hour": "peak_hour"
    })
)

# Build daily scheduling features
daily_schedule = (
    hourly.groupby("date")
    .agg(
        scheduled_room_hours=("scheduled_rooms", "sum"),
        max_scheduled_rooms=("scheduled_rooms", "max"),
        avg_temp=("outside_temp_c", "mean"),
        day_type=("day_type", "first")
    )
    .reset_index()
)

daily = daily_peak.merge(daily_schedule, on="date")

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").reset_index(drop=True)

# Previous-day peak features (past data only)
daily["peak_lag_1"] = daily["peak_kw"].shift(1)
daily["peak_lag_7"] = daily["peak_kw"].shift(7)
daily["peak_lag_14"] = daily["peak_kw"].shift(14)
daily["peak_mean_prev_7d"] = (
    daily["peak_kw"].shift(1).rolling(7).mean()
)

daily["dow"] = daily["date"].dt.dayofweek
daily["month"] = daily["date"].dt.month

daily_model_data = daily.dropna().reset_index(drop=True)

print("Hourly records:", len(hourly))
print("Daily records:", len(daily))
print("Usable model records:", len(daily_model_data))
print("Date range:", daily.date.min(), "->", daily.date.max())

display(daily_model_data.head())
display(daily_model_data.describe(include="all"))


Hourly records: 4368
Daily records: 182
Usable model records: 168
Date range: 2026-05-24 00:00:00 -> 2026-11-21 00:00:00


,date,peak_timestamp,peak_kw,peak_hour,scheduled_room_hours,max_scheduled_rooms,avg_temp,day_type,peak_lag_1,peak_lag_7,peak_lag_14,peak_mean_prev_7d,dow,month
0,2026-06-07,2026-06-07 15:00:00,41.896,15,120,17,28.491667,exam,13.489,41.798,30.870,36.600429,6,6
1,2026-06-08,2026-06-08 14:00:00,40.518,14,117,17,29.370833,exam,41.896,49.448,38.929,36.614429,0,6
2,2026-06-09,2026-06-09 13:00:00,57.797,13,144,20,29.850000,exam,40.518,52.415,51.627,35.338714,1,6
3,2026-06-10,2026-06-10 13:00:00,38.624,13,108,15,30.195833,exam,57.797,40.479,36.681,36.107571,2,6
4,2026-06-11,2026-06-11 13:00:00,46.555,13,131,18,30.629167,exam,38.624,50.134,49.833,35.842571,3,6


,date,peak_timestamp,peak_kw,peak_hour,scheduled_room_hours,max_scheduled_rooms,avg_temp,day_type,peak_lag_1,peak_lag_7,peak_lag_14,peak_mean_prev_7d,dow,month
count,168,168,168.000000,168.000000,168.000000,168.000000,168.000000,168,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,break,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57,NaN,NaN,NaN,NaN,NaN,NaN
mean,2026-08-29 12:00:00,2026-08-30 00:49:17.142856960,25.049494,12.821429,62.702381,9.297619,28.400868,NaN,25.081821,25.267595,25.260583,25.165032,3.000000,8.452381
min,2026-06-07 00:00:00,2026-06-07 15:00:00,7.976000,0.000000,0.000000,0.000000,21.100000,NaN,7.976000,7.976000,7.976000,14.588857,0.000000,6.000000
25%,2026-07-18 18:00:00,2026-07-19 05:30:00,9.721750,11.000000,0.000000,0.000000,25.594792,NaN,9.862500,9.862500,9.862500,16.914321,1.000000,7.000000
50%,2026-08-29 12:00:00,2026-08-30 00:30:00,20.499500,13.000000,45.000000,8.000000,29.025000,NaN,20.499500,20.499500,20.499500,26.054429,3.000000,8.000000
75%,2026-10-10 06:00:00,2026-10-10 19:15:00,40.679500,15.000000,130.000000,17.000000,30.684375,NaN,40.679500,41.420250,40.679500,32.755536,5.000000,10.000000
max,2026-11-21 00:00:00,2026-11-21 20:00:00,57.797000,20.000000,145.000000,21.000000,37.025000,NaN,57.797000,57.797000,57.797000,37.235714,6.000000,11.000000


In [2]:

# =====================================
# PEAK PREDICTION - DATA QUALITY AUDIT
# =====================================

print("1. TIMESTAMP QUALITY")

print("Duplicate timestamps:",
      hourly["timestamp"].duplicated().sum())

print("Missing hourly timestamps:",
      len(pd.date_range(
          hourly["timestamp"].min(),
          hourly["timestamp"].max(),
          freq="h"
      )) - hourly["timestamp"].nunique())


print("\n2. DAILY PEAK DISTRIBUTION")

print(daily["peak_kw"].describe())

print("Unique peak values:",
      daily["peak_kw"].nunique())

print("Repeated peak values:")
display(
    daily["peak_kw"]
    .value_counts()
    .head(10)
)


print("\n3. SCHEDULE FEATURES")

display(
    daily[
        [
            "scheduled_room_hours",
            "max_scheduled_rooms",
            "peak_kw"
        ]
    ].corr()
)


print("\n4. DAY TYPE DISTRIBUTION")

display(daily["day_type"].value_counts())


print("\n5. LAG FEATURE CORRELATION")

display(
    daily_model_data[
        [
            "peak_kw",
            "peak_lag_1",
            "peak_lag_7",
            "peak_lag_14",
            "peak_mean_prev_7d"
        ]
    ].corr()
)


print("\n6. HOURLY COVERAGE")

print("Rows per day:")
display(hourly.groupby("date").size().value_counts())

print("\nHourly records per day should normally be 24.")


1. TIMESTAMP QUALITY
Duplicate timestamps: 0
Missing hourly timestamps: 0

2. DAILY PEAK DISTRIBUTION
count    182.000000
mean      25.767033
std       15.160227
min        7.976000
25%        9.773500
50%       21.008000
75%       41.432750
max       57.797000
Name: peak_kw, dtype: float64
Unique peak values: 182
Repeated peak values:


,count
peak_kw,
30.870,1
38.929,1
51.627,1
36.681,1
49.833,1
8.847,1
8.295,1
41.798,1
49.448,1



3. SCHEDULE FEATURES


,scheduled_room_hours,max_scheduled_rooms,peak_kw
scheduled_room_hours,1.000000,0.989628,0.972674
max_scheduled_rooms,0.989628,1.000000,0.967129
peak_kw,0.972674,0.967129,1.000000



4. DAY TYPE DISTRIBUTION


,count
day_type,
break,57
weekend,52
regular,44
exam,25
holiday,4



5. LAG FEATURE CORRELATION


,peak_kw,peak_lag_1,peak_lag_7,peak_lag_14,peak_mean_prev_7d
peak_kw,1.000000,0.428697,0.828652,0.734416,0.483004
peak_lag_1,0.428697,1.000000,0.352724,0.303402,0.523987
peak_lag_7,0.828652,0.352724,1.000000,0.828586,0.536287
peak_lag_14,0.734416,0.303402,0.828586,1.000000,0.404747
peak_mean_prev_7d,0.483004,0.523987,0.536287,0.404747,1.000000



6. HOURLY COVERAGE
Rows per day:


,count
24,182



Hourly records per day should normally be 24.


In [3]:

# ==========================================
# PEAK DEMAND PREDICTION
# MODEL TRAINING & COMPARISON
# ==========================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Prepare features

data = daily_model_data.copy()
data = data.sort_values("date").reset_index(drop=True)

feature_cols = [
    "scheduled_room_hours",
    "max_scheduled_rooms",
    "day_type",
    "dow",
    "month",
    "peak_lag_1",
    "peak_lag_7",
    "peak_lag_14",
    "peak_mean_prev_7d"
]

target_col = "peak_kw"

X = pd.get_dummies(
    data[feature_cols],
    columns=["day_type"],
    dtype=int
)

y = data[target_col]

# 2. Chronological Train/Test Split

split_index = int(len(data) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("TRAINING DATA:", len(X_train))
print("TESTING DATA:", len(X_test))

print("Training period:",
      data["date"].iloc[0],
      "to",
      data["date"].iloc[split_index - 1])

print("Testing period:",
      data["date"].iloc[split_index],
      "to",
      data["date"].iloc[-1])


# 3. Define Models

models = {
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10))
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=3,
        random_state=42
    )
}

# LightGBM
try:
    from lightgbm import LGBMRegressor

    models["LightGBM"] = LGBMRegressor(
        n_estimators=150,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=7,
        min_child_samples=10,
        reg_alpha=1,
        reg_lambda=2,
        random_state=42,
        verbosity=-1
    )

except ImportError:
    print("LightGBM is not installed. Skipping it.")


# 4. Evaluation Function

results = []

# Baseline: Same weekday last week
baseline_predictions = data["peak_lag_7"].iloc[split_index:].values

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_wape = (
    np.sum(np.abs(y_test.values - baseline_predictions))
    / np.sum(np.abs(y_test.values))
) * 100

baseline_r2 = r2_score(y_test, baseline_predictions)

results.append({
    "model": "Weekly Baseline",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "WAPE_%": baseline_wape,
    "R2": baseline_r2
})


# 5. Train and Evaluate ML Models

trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    wape = (
        np.sum(np.abs(y_test.values - predictions))
        / np.sum(np.abs(y_test.values))
    ) * 100

    r2 = r2_score(y_test, predictions)

    results.append({
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_%": wape,
        "R2": r2
    })

    trained_models[name] = model


# 6. Compare Results

results_df = pd.DataFrame(results)

results_df = results_df.sort_values("WAPE_%")

print("\n========== MODEL COMPARISON ==========")

display(results_df.round(4))


# 7. Actual vs Predicted for Best Model

best_model_name = results_df.iloc[0]["model"]

if best_model_name == "Weekly Baseline":
    best_predictions = baseline_predictions
else:
    best_predictions = trained_models[best_model_name].predict(X_test)

comparison = pd.DataFrame({
    "date": data["date"].iloc[split_index:].values,
    "actual_peak_kw": y_test.values,
    "predicted_peak_kw": best_predictions
})

comparison["error_kw"] = (
    comparison["actual_peak_kw"] -
    comparison["predicted_peak_kw"]
)

print("\nBest Model:", best_model_name)

display(comparison.round(3))


TRAINING DATA: 134
TESTING DATA: 34
Training period: 2026-06-07 00:00:00 to 2026-10-18 00:00:00
Testing period: 2026-10-19 00:00:00 to 2026-11-21 00:00:00

Training Ridge Regression...

Training Random Forest...

Training LightGBM...

========== MODEL COMPARISON ==========


,model,MAE,RMSE,WAPE_%,R2
0,Weekly Baseline,1.7397,2.3195,5.5371,0.9764
3,LightGBM,1.8519,2.4496,5.8943,0.9737
2,Random Forest,2.4920,3.1045,7.9317,0.9578
1,Ridge Regression,2.5378,3.4659,8.0775,0.9474



Best Model: Weekly Baseline


,date,actual_peak_kw,predicted_peak_kw,error_kw
0,2026-10-19,34.847,34.505,0.342
1,2026-10-20,41.961,42.182,-0.221
2,2026-10-21,42.538,42.735,-0.197
3,2026-10-22,45.261,46.183,-0.922
4,2026-10-23,8.781,8.219,0.562
5,2026-10-24,8.096,9.568,-1.472
6,2026-10-25,42.040,47.197,-5.157
7,2026-10-26,37.024,34.847,2.177
8,2026-10-27,43.708,41.961,1.747
9,2026-10-28,44.278,42.538,1.740


In [4]:

# ==========================================
# PEAK PREDICTION - ERROR DIAGNOSTICS
# ==========================================

diagnostics = comparison.copy()

diagnostics["dow"] = pd.to_datetime(
    diagnostics["date"]
).dt.day_name()

diagnostics["baseline_error"] = (
    diagnostics["actual_peak_kw"]
    - diagnostics["predicted_peak_kw"]
).abs()

# LightGBM predictions on the same test period
lgbm_predictions = trained_models["LightGBM"].predict(X_test)

diagnostics["lgbm_prediction"] = lgbm_predictions

diagnostics["lgbm_error"] = (
    diagnostics["actual_peak_kw"]
    - diagnostics["lgbm_prediction"]
).abs()

# 1. Compare errors by weekday

weekday_results = (
    diagnostics.groupby("dow")
    .agg(
        days=("date", "count"),
        avg_actual_peak=("actual_peak_kw", "mean"),
        baseline_MAE=("baseline_error", "mean"),
        lightgbm_MAE=("lgbm_error", "mean")
    )
)

print("========== ERROR BY WEEKDAY ==========")
display(weekday_results.round(3))


# 2. Compare schedule and peak patterns

test_data = data.iloc[split_index:].copy()

schedule_analysis = test_data[
    [
        "date",
        "day_type",
        "scheduled_room_hours",
        "max_scheduled_rooms",
        "peak_kw",
        "peak_lag_7"
    ]
].copy()

schedule_analysis["dow"] = pd.to_datetime(
    schedule_analysis["date"]
).dt.day_name()

print("\n========== SCHEDULE vs PEAK ==========")

display(
    schedule_analysis.groupby("dow")[
        [
            "scheduled_room_hours",
            "max_scheduled_rooms",
            "peak_kw",
            "peak_lag_7"
        ]
    ].mean().round(2)
)


# 3. Identify days where ML performs better

diagnostics["better_model"] = np.where(
    diagnostics["lgbm_error"] < diagnostics["baseline_error"],
    "LightGBM",
    "Weekly Baseline"
)

print("\n========== MODEL WIN COUNT ==========")

display(
    diagnostics["better_model"].value_counts()
)


# 4. Check actual vs previous-week similarity

diagnostics["weekly_difference"] = (
    diagnostics["actual_peak_kw"]
    - diagnostics["predicted_peak_kw"]
)

print("\n========== WEEKLY PATTERN ==========")

print(
    "Mean absolute weekly difference:",
    round(diagnostics["weekly_difference"].abs().mean(), 3),
    "kW"
)

print(
    "Correlation with previous week:",
    round(
        diagnostics["actual_peak_kw"].corr(
            diagnostics["predicted_peak_kw"]
        ),
        4
    )
)


========== ERROR BY WEEKDAY ==========


,days,avg_actual_peak,baseline_MAE,lightgbm_MAE
dow,,,,
Friday,5,8.715,0.689,0.590
Monday,5,35.224,2.436,1.803
Saturday,5,8.221,0.591,1.192
Sunday,4,42.935,2.614,2.582
Thursday,5,44.176,1.210,1.420
Tuesday,5,41.191,3.152,3.442
Wednesday,5,41.771,1.660,2.081



========== SCHEDULE vs PEAK ==========


,scheduled_room_hours,max_scheduled_rooms,peak_kw,peak_lag_7
dow,,,,
Friday,0.0,0.0,8.72,8.62
Monday,130.0,17.0,35.22,35.38
Saturday,0.0,0.0,8.22,8.52
Sunday,141.0,21.0,42.94,43.20
Thursday,145.0,20.0,44.18,44.59
Tuesday,136.0,19.0,41.19,41.96
Wednesday,135.0,17.0,41.77,42.22



========== MODEL WIN COUNT ==========


,count
better_model,
Weekly Baseline,19
LightGBM,15



========== WEEKLY PATTERN ==========
Mean absolute weekly difference: 1.74 kW
Correlation with previous week: 0.9886


In [5]:

# ==========================================
# PEAK PREDICTION - WALK FORWARD BACKTESTING
# ==========================================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tscv = TimeSeriesSplit(n_splits=5, test_size=14)

backtest_predictions = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_te = X.iloc[test_idx]

    y_tr = y.iloc[train_idx]
    y_te = y.iloc[test_idx]

    test_dates = data["date"].iloc[test_idx].values

    # Weekly Baseline
    baseline_pred = data["peak_lag_7"].iloc[test_idx].values

    backtest_predictions.extend([
        {
            "fold": fold,
            "model": "Weekly Baseline",
            "date": date,
            "actual": actual,
            "predicted": pred
        }
        for date, actual, pred in zip(
            test_dates, y_te.values, baseline_pred
        )
    ])

    # Machine Learning models
    for name, original_model in models.items():

        model = clone(original_model)

        model.fit(X_tr, y_tr)

        predictions = model.predict(X_te)

        backtest_predictions.extend([
            {
                "fold": fold,
                "model": name,
                "date": date,
                "actual": actual,
                "predicted": pred
            }
            for date, actual, pred in zip(
                test_dates, y_te.values, predictions
            )
        ])


# Results
backtest_df = pd.DataFrame(backtest_predictions)

summary = []

for name, group in backtest_df.groupby("model"):

    actual = group["actual"].values
    predicted = group["predicted"].values

    summary.append({
        "model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_%": (
            np.abs(actual - predicted).sum()
            / np.abs(actual).sum()
        ) * 100,
        "R2": r2_score(actual, predicted)
    })

backtest_results = pd.DataFrame(summary)

backtest_results = backtest_results.sort_values("WAPE_%")

print("========== WALK-FORWARD BACKTEST RESULTS ==========")
display(backtest_results.round(4))


print("\n========== RESULTS BY FOLD ==========")

fold_results = (
    backtest_df.groupby(["fold", "model"])
    .apply(
        lambda g: pd.Series({
            "MAE": mean_absolute_error(
                g["actual"], g["predicted"]
            ),
            "WAPE_%": (
                np.abs(g["actual"] - g["predicted"]).sum()
                / np.abs(g["actual"]).sum()
            ) * 100
        }),
        include_groups=False
    )
    .reset_index()
)

display(fold_results.round(4))


========== WALK-FORWARD BACKTEST RESULTS ==========


,model,MAE,RMSE,WAPE_%,R2
0,LightGBM,2.1059,2.8863,6.8673,0.9667
2,Ridge Regression,2.4922,3.3435,8.1273,0.9554
1,Random Forest,2.8566,4.4018,9.3156,0.9226
3,Weekly Baseline,4.8755,9.5635,15.8992,0.6349



========== RESULTS BY FOLD ==========


,fold,model,MAE,WAPE_%
0,1,LightGBM,2.4319,9.6396
1,1,Random Forest,4.8772,19.3324
2,1,Ridge Regression,2.9745,11.7902
3,1,Weekly Baseline,10.8943,43.1828
4,2,LightGBM,2.6298,8.1608
5,2,Random Forest,3.0074,9.3325
6,2,Ridge Regression,3.1095,9.6492
7,2,Weekly Baseline,6.2549,19.4098
8,3,LightGBM,2.3044,7.0834
9,3,Random Forest,2.7615,8.4884


In [6]:

# ==========================================
# LIGHTGBM - WALK FORWARD FEATURE IMPORTANCE
# ==========================================

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance

tscv = TimeSeriesSplit(n_splits=5, test_size=14)

importance_results = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_te = X.iloc[test_idx]

    y_tr = y.iloc[train_idx]
    y_te = y.iloc[test_idx]

    model = clone(models["LightGBM"])

    model.fit(X_tr, y_tr)

    result = permutation_importance(
        model,
        X_te,
        y_te,
        scoring="neg_mean_absolute_error",
        n_repeats=10,
        random_state=42
    )

    fold_importance = pd.DataFrame({
        "feature": X.columns,
        "importance": result.importances_mean,
        "fold": fold
    })

    importance_results.append(fold_importance)


importance_df = pd.concat(
    importance_results,
    ignore_index=True
)

importance_summary = (
    importance_df.groupby("feature")
    .agg(
        mean_importance=("importance", "mean"),
        std_importance=("importance", "std")
    )
    .sort_values("mean_importance", ascending=False)
)

print("========== WALK-FORWARD FEATURE IMPORTANCE ==========")

display(importance_summary.round(4))


========== WALK-FORWARD FEATURE IMPORTANCE ==========


,mean_importance,std_importance
feature,,
scheduled_room_hours,10.2703,0.5494
max_scheduled_rooms,3.3884,0.4689
peak_lag_14,0.1645,0.1101
dow,0.0701,0.1235
peak_lag_7,0.0562,0.2069
peak_mean_prev_7d,0.0062,0.0167
day_type_break,0.0006,0.0013
day_type_holiday,0.0000,0.0000
day_type_exam,0.0000,0.0000


In [7]:

# ==========================================
# PEAK PREDICTION - FEATURE ABLATION
# ==========================================

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Feature groups

schedule_features = [
    "scheduled_room_hours",
    "max_scheduled_rooms",
    "dow",
    "month"
] + [c for c in X.columns if c.startswith("day_type_")]

history_features = [
    "peak_lag_1",
    "peak_lag_7",
    "peak_lag_14",
    "peak_mean_prev_7d",
    "dow",
    "month"
]

feature_groups = {
    "Schedule Only": schedule_features,
    "Historical Only": history_features,
    "Combined": list(X.columns)
}

# Walk-forward validation

tscv = TimeSeriesSplit(n_splits=5, test_size=14)

ablation_predictions = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):

    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]

    for experiment, features in feature_groups.items():

        X_train_fold = X.iloc[train_idx][features]
        X_test_fold = X.iloc[test_idx][features]

        model = clone(models["LightGBM"])

        model.fit(X_train_fold, y_train_fold)

        predictions = model.predict(X_test_fold)

        for date, actual, predicted in zip(
            data["date"].iloc[test_idx],
            y_test_fold,
            predictions
        ):
            ablation_predictions.append({
                "fold": fold,
                "experiment": experiment,
                "date": date,
                "actual": actual,
                "predicted": predicted
            })


# Aggregate results

ablation_df = pd.DataFrame(ablation_predictions)

ablation_results = []

for name, group in ablation_df.groupby("experiment"):

    actual = group["actual"].values
    predicted = group["predicted"].values

    ablation_results.append({
        "experiment": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "WAPE_%": (
            np.abs(actual - predicted).sum()
            / np.abs(actual).sum()
        ) * 100,
        "R2": r2_score(actual, predicted)
    })


ablation_results_df = pd.DataFrame(ablation_results)

ablation_results_df = ablation_results_df.sort_values("WAPE_%")

print("========== FEATURE ABLATION RESULTS ==========")

display(ablation_results_df.round(4))


========== FEATURE ABLATION RESULTS ==========


,experiment,MAE,RMSE,WAPE_%,R2
2,Schedule Only,2.0896,2.9157,6.8142,0.9661
0,Combined,2.1059,2.8863,6.8673,0.9667
1,Historical Only,5.6423,9.0694,18.3999,0.6716


In [8]:

# ==========================================
# SCHEDULE FEATURE ABLATION
# Walk-Forward Backtesting - 5 Folds
# ==========================================

import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Prepare dataset
ablation_data = daily_model_data.copy()
ablation_data = ablation_data.sort_index()

target = ablation_data["peak_kw"]

# 2. Create Day Type features
day_type_encoded = pd.get_dummies(
    ablation_data["day_type"],
    prefix="day_type",
    dtype=int
)

# 3. Define feature groups
feature_sets = {
    "A - Schedule Hours Only": [
        "scheduled_room_hours"
    ],

    "B - Schedule Numeric": [
        "scheduled_room_hours",
        "max_scheduled_rooms"
    ],

    "C - Schedule + Day of Week": [
        "scheduled_room_hours",
        "max_scheduled_rooms",
        "dow"
    ],

    "D - Full Schedule Features": [
        "scheduled_room_hours",
        "max_scheduled_rooms",
        "dow",
        "month"
    ] + list(day_type_encoded.columns)
}

# 4. Build complete feature matrix
X_all = pd.concat(
    [
        ablation_data[
            ["scheduled_room_hours", "max_scheduled_rooms", "dow", "month"]
        ],
        day_type_encoded
    ],
    axis=1
)

# 5. Walk-Forward Backtesting
tscv = TimeSeriesSplit(n_splits=5, test_size=14)

results = []

for experiment_name, features in feature_sets.items():

    X = X_all[features]

    actual_all = []
    predicted_all = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = target.iloc[train_idx]
        y_test = target.iloc[test_idx]

        model = LGBMRegressor(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=7,
            min_child_samples=10,
            reg_alpha=1,
            reg_lambda=2,
            random_state=42,
            verbosity=-1
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        actual_all.extend(y_test.tolist())
        predicted_all.extend(predictions.tolist())

    # 6. Calculate metrics across all folds
    mae = mean_absolute_error(actual_all, predicted_all)

    rmse = np.sqrt(
        mean_squared_error(actual_all, predicted_all)
    )

    wape = (
        np.sum(np.abs(np.array(actual_all) - np.array(predicted_all)))
        / np.sum(np.abs(actual_all))
    ) * 100

    r2 = r2_score(actual_all, predicted_all)

    results.append({
        "experiment": experiment_name,
        "features_count": len(features),
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_%": wape,
        "R2": r2
    })

# 7. Display results
schedule_ablation_results = pd.DataFrame(results)

schedule_ablation_results = schedule_ablation_results.sort_values(
    by="MAE"
).reset_index(drop=True)

print("\n========== SCHEDULE FEATURE ABLATION RESULTS ==========\n")

display(
    schedule_ablation_results.round(4)
)




========== SCHEDULE FEATURE ABLATION RESULTS ==========



,experiment,features_count,MAE,RMSE,WAPE_%,R2
0,D - Full Schedule Features,9,2.0896,2.9157,6.8142,0.9661
1,C - Schedule + Day of Week,3,2.7864,3.7804,9.0867,0.9429
2,B - Schedule Numeric,2,2.8652,3.8425,9.3436,0.9411
3,A - Schedule Hours Only,1,2.9969,4.0327,9.7729,0.9351


In [9]:

# ==========================================
# CALENDAR FEATURE ABLATION
# Walk-Forward Backtesting - 5 Folds
# ==========================================

import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Prepare data
data = daily_model_data.copy().sort_index()

y = data["peak_kw"]

# 2. Encode day_type
day_type_encoded = pd.get_dummies(
    data["day_type"],
    prefix="day_type",
    dtype=int
)

# 3. Build feature matrix
X = pd.concat(
    [
        data[
            [
                "scheduled_room_hours",
                "max_scheduled_rooms",
                "dow",
                "month"
            ]
        ],
        day_type_encoded
    ],
    axis=1
)

# 4. Define feature groups
numeric_features = [
    "scheduled_room_hours",
    "max_scheduled_rooms"
]

dow_feature = ["dow"]
month_feature = ["month"]
day_type_features = list(day_type_encoded.columns)

feature_sets = {
    "A - Numeric + DOW":
        numeric_features + dow_feature,

    "B - Numeric + Month":
        numeric_features + month_feature,

    "C - Numeric + Day Type":
        numeric_features + day_type_features,

    "D - Numeric + DOW + Month":
        numeric_features + dow_feature + month_feature,

    "E - Numeric + DOW + Day Type":
        numeric_features + dow_feature + day_type_features,

    "F - Full Calendar Features":
        numeric_features
        + dow_feature
        + month_feature
        + day_type_features
}

# 5. Walk-forward validation
tscv = TimeSeriesSplit(n_splits=5, test_size=14)

results = []
fold_results = []

for experiment_name, features in feature_sets.items():

    X_exp = X[features]

    actual_all = []
    predicted_all = []

    for fold, (train_idx, test_idx) in enumerate(
        tscv.split(X_exp), start=1
    ):

        X_train = X_exp.iloc[train_idx]
        X_test = X_exp.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = LGBMRegressor(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=7,
            min_child_samples=10,
            reg_alpha=1,
            reg_lambda=2,
            random_state=42,
            verbosity=-1
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        actual_all.extend(y_test.tolist())
        predicted_all.extend(predictions.tolist())

        fold_results.append({
            "experiment": experiment_name,
            "fold": fold,
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "WAPE_%": (
                np.sum(np.abs(y_test.values - predictions))
                / np.sum(np.abs(y_test.values))
            ) * 100
        })

    # 6. Overall metrics across all folds
    actual_all = np.array(actual_all)
    predicted_all = np.array(predicted_all)

    results.append({
        "experiment": experiment_name,
        "features_count": len(features),
        "MAE": mean_absolute_error(
            actual_all, predicted_all
        ),
        "RMSE": np.sqrt(
            mean_squared_error(actual_all, predicted_all)
        ),
        "WAPE_%": (
            np.sum(np.abs(actual_all - predicted_all))
            / np.sum(np.abs(actual_all))
        ) * 100,
        "R2": r2_score(actual_all, predicted_all)
    })

# 7. Display overall results
calendar_ablation_results = pd.DataFrame(results)

calendar_ablation_results = calendar_ablation_results.sort_values(
    "MAE"
).reset_index(drop=True)

print("\n========== CALENDAR FEATURE ABLATION ==========\n")
display(calendar_ablation_results.round(4))

# 8. Display fold-level results
calendar_fold_results = pd.DataFrame(fold_results)

print("\n========== FOLD-LEVEL RESULTS ==========\n")
display(
    calendar_fold_results.round(4)
)



========== CALENDAR FEATURE ABLATION ==========



,experiment,features_count,MAE,RMSE,WAPE_%,R2
0,F - Full Calendar Features,9,2.0896,2.9157,6.8142,0.9661
1,D - Numeric + DOW + Month,4,2.0948,2.9195,6.8312,0.9660
2,B - Numeric + Month,3,2.1269,3.0184,6.9360,0.9636
3,E - Numeric + DOW + Day Type,8,2.2285,3.1253,7.2673,0.9610
4,C - Numeric + Day Type,7,2.2658,3.1836,7.3889,0.9595
5,A - Numeric + DOW,3,2.7864,3.7804,9.0867,0.9429



========== FOLD-LEVEL RESULTS ==========



,experiment,fold,MAE,RMSE,WAPE_%
0,A - Numeric + DOW,1,2.3863,4.1308,9.4588
1,A - Numeric + DOW,2,2.4763,3.3836,7.6843
2,A - Numeric + DOW,3,3.3984,4.1674,10.4462
3,A - Numeric + DOW,4,3.4232,4.1321,10.8693
4,A - Numeric + DOW,5,2.2480,2.9164,7.0592
5,B - Numeric + Month,1,2.6186,4.1616,10.3794
6,B - Numeric + Month,2,2.4944,3.3170,7.7406
7,B - Numeric + Month,3,2.3703,2.9187,7.2860
8,B - Numeric + Month,4,2.0273,2.4524,6.4371
9,B - Numeric + Month,5,1.1240,1.6427,3.5297


In [10]:
calendar_ablation_results.round(4)

,experiment,features_count,MAE,RMSE,WAPE_%,R2
0,F - Full Calendar Features,9,2.0896,2.9157,6.8142,0.9661
1,D - Numeric + DOW + Month,4,2.0948,2.9195,6.8312,0.9660
2,B - Numeric + Month,3,2.1269,3.0184,6.9360,0.9636
3,E - Numeric + DOW + Day Type,8,2.2285,3.1253,7.2673,0.9610
4,C - Numeric + Day Type,7,2.2658,3.1836,7.3889,0.9595
5,A - Numeric + DOW,3,2.7864,3.7804,9.0867,0.9429


In [11]:

# ==========================================
# FINAL PEAK DEMAND MODEL
# Training + Saving Model Artifact
# ==========================================

import os
import json
import joblib
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor

# 1. Define final features
FINAL_FEATURES = [
    "scheduled_room_hours",
    "max_scheduled_rooms",
    "dow",
    "month"
]

TARGET = "peak_kw"

# 2. Prepare final training data
final_data = daily_model_data.copy().sort_index()

X_final = final_data[FINAL_FEATURES].copy()
y_final = final_data[TARGET].copy()

# 3. Train final LightGBM model
final_peak_model = LGBMRegressor(
    n_estimators=150,
    learning_rate=0.03,
    max_depth=3,
    num_leaves=7,
    min_child_samples=10,
    reg_alpha=1,
    reg_lambda=2,
    random_state=42,
    verbosity=-1
)

final_peak_model.fit(X_final, y_final)

# 4. Create artifact directory
ARTIFACT_DIR = "models/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# 5. Save model bundle
model_bundle = {
    "model": final_peak_model,
    "features": FINAL_FEATURES,
    "target": TARGET,
    "model_name": "LightGBM Peak Demand Prediction",
    "model_version": "1.0",
    "training_rows": len(X_final),
    "training_start": str(final_data.index.min()),
    "training_end": str(final_data.index.max()),
    "backtesting_metrics": {
        "MAE": 2.0948,
        "RMSE": 2.9195,
        "WAPE_%": 6.8312,
        "R2": 0.9660
    }
}

MODEL_PATH = os.path.join(
    ARTIFACT_DIR,
    "peak_demand_model.joblib"
)

joblib.dump(model_bundle, MODEL_PATH)

print("========== FINAL MODEL SAVED ==========")
print(f"Model: {model_bundle['model_name']}")
print(f"Training rows: {len(X_final)}")
print(f"Training period: {final_data.index.min()} -> {final_data.index.max()}")
print(f"Features: {FINAL_FEATURES}")
print(f"Saved to: {MODEL_PATH}")


========== FINAL MODEL SAVED ==========
Model: LightGBM Peak Demand Prediction
Training rows: 168
Training period: 0 -> 167
Features: ['scheduled_room_hours', 'max_scheduled_rooms', 'dow', 'month']
Saved to: models/artifacts/peak_demand_model.joblib


In [12]:

# ==========================================
# LOAD AND TEST SAVED MODEL
# ==========================================

loaded_bundle = joblib.load(MODEL_PATH)

loaded_model = loaded_bundle["model"]
loaded_features = loaded_bundle["features"]

# Test using the last available day's features
test_input = X_final.iloc[[-1]][loaded_features]

prediction = loaded_model.predict(test_input)[0]

actual = y_final.iloc[-1]

print("========== MODEL TEST ==========")
print(f"Actual Peak Demand: {actual:.2f} kW")
print(f"Predicted Peak Demand: {prediction:.2f} kW")
print(f"Absolute Error: {abs(actual - prediction):.2f} kW")

print("\nModel loaded successfully!")


========== MODEL TEST ==========
Actual Peak Demand: 8.06 kW
Predicted Peak Demand: 9.26 kW
Absolute Error: 1.20 kW

Model loaded successfully!


In [13]:

print("Index type:", type(daily_model_data.index))
print("Index sample:", daily_model_data.index[:5].tolist())

print("\nAvailable columns:")
print(daily_model_data.columns.tolist())

print("\nDate column candidates:")
for col in daily_model_data.columns:
    if "date" in col.lower() or "time" in col.lower():
        print(col, daily_model_data[col].head().tolist())


Index type: <class 'pandas.core.indexes.range.RangeIndex'>
Index sample: [0, 1, 2, 3, 4]

Available columns:
['date', 'peak_timestamp', 'peak_kw', 'peak_hour', 'scheduled_room_hours', 'max_scheduled_rooms', 'avg_temp', 'day_type', 'peak_lag_1', 'peak_lag_7', 'peak_lag_14', 'peak_mean_prev_7d', 'dow', 'month']

Date column candidates:
date [Timestamp('2026-06-07 00:00:00'), Timestamp('2026-06-08 00:00:00'), Timestamp('2026-06-09 00:00:00'), Timestamp('2026-06-10 00:00:00'), Timestamp('2026-06-11 00:00:00')]
peak_timestamp [Timestamp('2026-06-07 15:00:00'), Timestamp('2026-06-08 14:00:00'), Timestamp('2026-06-09 13:00:00'), Timestamp('2026-06-10 13:00:00'), Timestamp('2026-06-11 13:00:00')]


In [14]:

# ==========================================
# FIX MODEL METADATA
# ==========================================

# Read actual dates from the date column
training_dates = pd.to_datetime(final_data["date"])

# Update model metadata
model_bundle["training_start"] = str(training_dates.min().date())
model_bundle["training_end"] = str(training_dates.max().date())
model_bundle["training_rows"] = len(final_data)

# Add target unit and training information
model_bundle["target_unit"] = "kW"
model_bundle["features"] = FINAL_FEATURES

# Save updated artifact
joblib.dump(model_bundle, MODEL_PATH)

# Verify saved metadata
verified_bundle = joblib.load(MODEL_PATH)

print("========== VERIFIED MODEL METADATA ==========")
print("Model:", verified_bundle["model_name"])
print("Training rows:", verified_bundle["training_rows"])
print("Training start:", verified_bundle["training_start"])
print("Training end:", verified_bundle["training_end"])
print("Features:", verified_bundle["features"])
print("Target:", verified_bundle["target"], verified_bundle["target_unit"])
print("Artifact:", MODEL_PATH)


========== VERIFIED MODEL METADATA ==========
Model: LightGBM Peak Demand Prediction
Training rows: 168
Training start: 2026-06-07
Training end: 2026-11-21
Features: ['scheduled_room_hours', 'max_scheduled_rooms', 'dow', 'month']
Target: peak_kw kW
Artifact: models/artifacts/peak_demand_model.joblib


In [15]:
import os

print("Current directory:", os.getcwd())
print("Absolute model path:", os.path.abspath(MODEL_PATH))
print("File exists:", os.path.exists(MODEL_PATH))
print("File size:", os.path.getsize(MODEL_PATH), "bytes")


Current directory: /content
Absolute model path: /content/models/artifacts/peak_demand_model.joblib
File exists: True
File size: 109381 bytes
